In [ ]:
import sys
sys.path.append("../")

# autoreload
%load_ext autoreload
%autoreload 2

import pulse_simulator as ps

import numpy as np
import matplotlib.pyplot as plt
import jax

# qiskit
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.quantum_info import process_fidelity
import qiskit_dynamics

# jax stuff
jax.config.update("jax_enable_x64", True)
qiskit_dynamics.array.Array.set_default_backend("jax")

# Check if GPU is available
print("jax backend: ", jax.default_backend())

In [ ]:
num_qubits = 3

# Basis gates
basis_gates = ["id", "rz", "sx", "x", "ecr"]

# Linear connectivity (with 2QB gates in both directions)
coupling_map = (
    [[i, i + 1] for i in range(num_qubits - 1)] 
    + [[i + 1, i] for i in range(num_qubits - 1)]
)

backend = GenericBackendV2(
    num_qubits=num_qubits, 
    basis_gates=basis_gates,
    coupling_map=coupling_map,
)

# Select subset of registers to simulate
registers = [0, 1]

# backend.coupling_map.draw()

In [ ]:
ns = 1e-9
dt_simulation =  backend.dt / ns
desired_1qb_samples = 120
duration_1qb = ps.qiskit_closest_multiple_of_16(desired_1qb_samples)

# default_1qb_gateset = ps.default_single_qubit_gateset(
#     ["X1", "X2", "SX1", "SX2", "ID1", "ID2"], duration_1qb, dt_simulation
# )

default_1qb_gateset = ps.default_single_qubit_gateset(
    ["X1", "SX1", "ID1"], duration_1qb, dt_simulation
)

default_1qb_gateset["X1"].draw()

In [ ]:
from qiskit.circuit.random import random_circuit

depth = 4
seed = 17
# This circuit will only use 1QB gates
rand_circ = random_circuit(len(registers), seed, max_operands=1)

In [ ]:
rand_circ.draw('mpl')

In [ ]:
from qiskit import transpile

transpile(rand_circ, backend, optimization_level=0).draw('mpl')
# transpile(rand_circ, backend, optimization_level=1).draw('mpl')

In [ ]:
# Build the pulse simulator

# Manually pass parameters. These are the detuning parameters (after changing units and without 2pi).
# Try changing them to 0.0 or 0.01 or 0.001, and see how the fidelity changes.

simulation_vars = [{
    "deltaq0": 0.001,
    "deltaq1": 0.001,
}]

simulator = ps.Simulator(registers, backend, simulation_vars=simulation_vars)

In [ ]:
simulator._simulation_vars

In [ ]:
# Add some errors to make it interesting!
res = simulator.simulate_circuit(
    rand_circ, 
    default_1qb_gateset, 
    None, # default 2QB gateset
    dt_simulation, 
    include_crosstalk=False, 
    include_dissipation=False,
    gpu=True,
    coloring = lambda q: 1,
)

In [ ]:
moments = simulator.get_moments()
moments

In [ ]:
# Let's check in on the simulation after each gate
moments = simulator.get_moments()
measure_times = np.cumsum(
    ps.durations_of_moments(moments, duration_1qb, 0.0)
) * backend.dt

# These were the expected outcomes
U_goals = [] 
for i in range(len(moments)):
    U_goals.append(ps.build_operator(moments[0:i+1], registers))

# How did we do?
fidelities = np.array([process_fidelity(R, U) for R, U in zip(res[1:], U_goals)])

# Plot the result
fig, ax = plt.subplots(1)
ax.plot(measure_times[1:], 1 - fidelities[1:], marker="o")
ax.set_xlabel("Elapsed time (s)")
ax.set_ylabel("Infidelity")
ax.set_yscale("log")
ax.set_ylim(1e-6, 1)

In [ ]:
simulator.build_circuit(insert_barriers=True).draw('mpl')